# 04 — Anotación, Fase 1: calibración sobre ES2015

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`03-jss-eda.ipynb`](03-jss-eda.ipynb) (conteos de la serie y partición
desarrollo / evaluación) · [manual de anotación](../docs/anotacion/manual-anotacion-oe1.md)

---

> **Aviso: el kappa de esta fase es diagnóstico y NO se reporta en la tesis.**
> Mide si las dos personas que anotan entendieron lo mismo de la guía *antes* de tocar
> ninguna serie de evaluación. El kappa reportable es el de la Fase 2
> (notebook 05, `05-jss-anotacion-f2-control.ipynb`, pendiente).

## Qué hace este notebook

Es el registro de la **Fase 1** de la anotación humana de OE1: dos anotadores (`gm` y `gv`)
etiquetaron por separado la serie ES2015, y `jss` adjudicó los desacuerdos. El notebook
reconstruye, paso a paso:

1. Qué es la Fase 1 y por qué se hace sobre ES2015 (§0).
2. Qué insumos entran y por qué etapas pasan hasta quedar adjudicados (§1).
3. Cuánto coincidieron los dos anotadores al juzgar cada decisión (§2) y cada par (§3).
4. Qué resolvió la adjudicación y con qué razones (§4).
5. Qué vacíos de la guía reveló la calibración y qué se decidió en cada uno (§5).
6. Qué cambia para la Fase 2 y qué límites tiene esta fase (§6).

## Qué NO hace este notebook

- **No reporta un kappa para la tesis.** Los valores de §2 y §3 son diagnósticos.
- **No modifica ninguna etiqueta.** Las etiquetas son humanas: el notebook lee los CSV ya
  versionados y los muestra. Ni la máquina ni este notebook eligen entre dos anotadores.
- **No calcula en el notebook.** Toda cifra viene de una función testeada de
  `src/afg/annotation/` ([`CONTRIBUTING.md` §7.1](../CONTRIBUTING.md)); el notebook importa,
  llama y explica.

In [1]:
import textwrap
import warnings

import pandas as pd

from afg.annotation.adjudication import validate_adjudication
from afg.annotation.agreement import (
    annotator_csvs,
    compute_series_agreement,
    series_axis_profiles,
)
from afg.annotation.calibration import (
    adjudicated_pairs,
    compuesta_children,
    decision_agreement,
    final_rows,
    final_status_counts,
    resolved_disagreements,
    side_taken_counts,
    status_contingency,
)
from afg.annotation.workspace import TaskKind, WorkspacePaths
from afg.shared.paths import PROJECT_ROOT

SERIES = "ES2015"
# Cohen's kappa is undefined (NaN) when both annotators use a single label; scikit-learn warns
# about it on every such axis. The NaN is explained in section 3.4, so the warnings are muted.
warnings.filterwarnings("ignore", module="sklearn")

PATHS = WorkspacePaths()  # default locations under data/processed/

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)


def show_razon(record, fields):
    """Print one adjudication record with the adjudicator's reason verbatim.

    The reason is never edited here: it is the adjudicator's own wording and part of the
    record. Only the layout (wrapping) is presentational.
    """
    head = " | ".join(f"{name}={record[name]}" for name in fields)
    print(head)
    print(
        textwrap.fill(
            f"razon: {record['razon']}",
            width=110,
            initial_indent="    ",
            subsequent_indent="           ",
        )
    )
    print()


# Guard: these files are committed, so a clone has them; fail clearly if any is missing.
missing = [
    p.name
    for p in (
        PATHS.adjudicated(TaskKind.DECISIONS, SERIES),
        PATHS.adjudicated(TaskKind.CANDIDATES, SERIES),
        *annotator_csvs(PATHS.decisions_dir, SERIES, "decisions"),
        *annotator_csvs(PATHS.relations_dir, SERIES, "candidates"),
    )
    if not p.exists()
]
if missing or len(annotator_csvs(PATHS.relations_dir, SERIES, "candidates")) != 2:
    raise SystemExit(f"Faltan archivos de la Fase 1 para {SERIES}: {missing}")
print(
    f"Serie {SERIES}; datos en {PATHS.decisions_dir.relative_to(PROJECT_ROOT)}/ y "
    f"{PATHS.relations_dir.relative_to(PROJECT_ROOT)}/"
)

Serie ES2015; datos en data/processed/decisions/ y data/processed/relations/


---

# 0. Qué es la Fase 1 y por qué ES2015

## 0.1 Para qué sirve

La anotación de OE1 produce el conjunto de referencia contra el que se evalúan los
experimentos. Antes de anotar las series que se *reportan*, hay que comprobar que las dos
personas que anotan **entienden la guía de la misma manera**. La Fase 1 es esa comprobación:
`gm` y `gv` anotan la misma serie, por separado y sin conversar sobre casos, y se mide cuánto
coinciden. Si coinciden poco, el problema suele estar en la guía y no en las personas, y se
corrige la guía *antes* de pasar a la Fase 2.

Las tareas que anotan son dos:

- **Tarea A (decisiones).** Para cada frase de decisión extraída por la máquina, asignar un
  estado: `decision`, `no_decision`, `compuesta`, `sin_soporte` o `descartar`
  (manual §2). Si es una decisión, se anotan su objeto y su contenido.
- **Tarea B (pares).** Para cada par de decisiones de reuniones distintas que el filtro de
  bloqueo marcó como candidato, asignar la relación temporal (`reafirma`, `refina`, ...) y
  si la dirección es correcta (manual §4 y §6).

## 0.2 Por qué ES2015 y no una serie de control

La elección es de costo. ES2015 es una **serie de desarrollo**: su contenido ya se leyó
completo durante el piloto, de modo que
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md) permite citarlo y
gastarla en calibrar no contamina nada. Calibrar sobre una serie de control habría sido
caro: el primer kappa —el peor, el que mide "todavía no se entiende la guía"— habría sido
un kappa **reportable**, y habría que reportarlo.

> **Regla de la fase.** Un kappa calculado sobre una serie de desarrollo es una herramienta
> de trabajo. Su única pregunta es *¿entendimos los dos lo mismo?* No entra a la tesis.

El fundamento del reparto está en
[`docs/anotacion/asignacion/README.md`](../docs/anotacion/asignacion/README.md) (§ Fase 1).

---

# 1. Insumos y flujo

## 1.1 De la máquina a la referencia adjudicada

Cada flecha es una etapa con su archivo. Las columnas de máquina nunca se tocan; las
etiquetas las pone siempre una persona.

| Etapa | Quién | Archivo (en `data/processed/`) |
|---|---|---|
| Frases de decisión extraídas, con su evidencia | máquina | `decisions/ES2015.decisions.csv` |
| Pares candidatos de reuniones distintas | máquina (filtro de bloqueo) | `relations/ES2015.candidates.csv` |
| Tarea A, por separado | `gm` · `gv` | `decisions/ES2015.decisions.gm.csv` · `.gv.csv` |
| Tarea B, por separado | `gm` · `gv` | `relations/ES2015.candidates.gm.csv` · `.gv.csv` |
| Acuerdo entre anotadores | función `compute_series_agreement` | `reports/tables/ES2015.agreement.csv` |
| Adjudicación de los desacuerdos | `jss` | `decisions/ES2015.decisions.adjudicated.csv` · `relations/ES2015.candidates.adjudicated.csv` |
| Verificación de que el CSV adjudicado está completo | `afg gold validate-adjudication` | (sin archivo; código de salida) |

Dos reglas de diseño sostienen la validez de lo que sigue:

- **Independencia.** Durante las Fases 1 y 2, `gm` y `gv` no comparan etiquetas ni discuten
  casos concretos antes de la sesión. Si lo hicieran, el kappa dejaría de medir acuerdo y
  pasaría a medir conversación.
- **Quien adjudica no anota.** `jss` no etiqueta ninguna fila; solo resuelve donde los dos
  anotadores difieren. Si adjudicara sus propias etiquetas, las finales quedarían sesgadas
  hacia uno de los tres.

Quién cumplió cada etapa se puede comprobar en las ramas y los PR del repositorio; aquí se
verifica lo que se puede verificar con los datos: que existan los archivos y que la
adjudicación esté completa.

In [2]:
# The adjudication is "complete" when the validator finds no issue in either CSV: every row
# has a closed-set verdict, every disagreement has a reason, every compuesta has its children.
issues = validate_adjudication(SERIES, paths=PATHS)
print(f"Problemas encontrados por validate-adjudication: {len(issues)}")

# Size of each stage (rows), straight from the committed files.
sizes = pd.DataFrame(
    [
        ("decisions (máquina)", len(pd.read_csv(PATHS.source(TaskKind.DECISIONS, SERIES)))),
        ("candidates (máquina)", len(pd.read_csv(PATHS.source(TaskKind.CANDIDATES, SERIES)))),
        (
            "decisions adjudicado (base + hijas)",
            len(pd.read_csv(PATHS.adjudicated(TaskKind.DECISIONS, SERIES))),
        ),
        (
            "candidates adjudicado",
            len(pd.read_csv(PATHS.adjudicated(TaskKind.CANDIDATES, SERIES))),
        ),
    ],
    columns=["archivo", "filas"],
)
display(sizes)

Problemas encontrados por validate-adjudication: 0


,archivo,filas
0,decisions (máquina),29
1,candidates (máquina),3
2,decisions adjudicado (base + hijas),34
3,candidates adjudicado,3


**Lectura.** Las 29 decisiones base de la máquina pasan a 34 filas adjudicadas porque dos
filas se declararon `compuesta` y se dividieron en filas hijas (§4.2); los 3 pares quedan en 3.

---

# 2. Acuerdo en decisiones (Tarea A)

## 2.1 Qué se mide

Para cada una de las 29 filas base se compara el `status` que puso `gm` con el que puso `gv`.
Se reportan el **acuerdo crudo** (porcentaje de filas con el mismo estado) y el **kappa de
Cohen**, que descuenta el acuerdo esperable por azar.

> Este kappa de estado **no** es uno de los tres kappas de la Tarea D (existencia, tipo y
> dirección, manual §6), que se calculan sobre pares y se ven en §3. Es un diagnóstico
> adicional de esta fase. Como todo lo de esta fase, no entra a la tesis.

In [3]:
agreement = decision_agreement(SERIES, paths=PATHS)
a, b = agreement.annotator_a, agreement.annotator_b
print(f"Anotadores: {a} y {b}")
print(f"Filas base comparadas:           {agreement.n_items}")
print(
    f"Mismo estado:                    {agreement.n_status_agree} "
    f"({agreement.percent_agreement:.1%})"
)
print(f"Filas con acuerdo = si:          {agreement.n_acuerdo}")
print(f"Kappa de Cohen sobre el estado:  {agreement.status_kappa:.3f}  (DIAGNÓSTICO)")

Anotadores: gm y gv
Filas base comparadas:           29
Mismo estado:                    13 (44.8%)
Filas con acuerdo = si:          13
Kappa de Cohen sobre el estado:  0.094  (DIAGNÓSTICO)


## 2.2 Dónde coincidieron y dónde no

La tabla cruza lo que puso cada anotador (filas: `gm`; columnas: `gv`). La diagonal es el
acuerdo.

In [4]:
records = status_contingency(SERIES, paths=PATHS)
contingency = (
    pd.DataFrame(records, columns=[f"{a} (filas)", f"{b} (columnas)", "n"])
    .pivot(index=f"{a} (filas)", columns=f"{b} (columnas)", values="n")
    .fillna(0)
    .astype(int)
)
display(contingency)

gv (columnas),compuesta,decision,no_decision,sin_soporte
gm (filas),,,,
compuesta,0,1,0,0
decision,1,11,6,8
no_decision,0,0,1,0
sin_soporte,0,0,0,1


**Lectura.**

- **Hay 13 coincidencias en 29 filas (44,8 %) y el kappa es 0,094**, es decir, apenas por
  encima de lo que daría el azar. Con un diagnóstico así, la lectura honesta es que la guía
  vigente al comenzar la fase **no bastaba** para que dos personas llegaran al mismo juicio.
  Para eso existe la Fase 1.
- **El desacuerdo tiene dirección.** Casi todos los desacuerdos son de la forma `gm = decision`
  frente a `gv` más estricto: 6 filas en que `gv` ve `no_decision` y 8 en que ve
  `sin_soporte`, además de 2 filas donde una persona ve `compuesta` y la otra no. `gm` casi
  siempre acepta la frase como decisión; `gv` exige más a la evidencia. No es un error de
  ninguno: es la diferencia de umbral que la guía debía fijar (§5).
- **El kappa es bajo en parte por la forma de la tabla**: una categoría (`decision`)
  concentra casi todo el peso de `gm`, y cuando una categoría domina los marginales el kappa
  se deprime respecto del acuerdo crudo. Es un límite conocido de la estadística y
  una razón más para no reportarla aquí.

---

# 3. Acuerdo en pares (Tarea B)

## 3.1 Por qué solo hay 3 pares

El filtro de bloqueo deja pasar, de los 311 pares entre reuniones distintas de ES2015
(notebook [03](03-jss-eda.ipynb), §1.4), solo **3**: una tasa de 0,96 %, una de las más bajas
de la muestra (el rango va de 0,7 % a 13,3 %). Es una consecuencia del filtro, no una decisión de la anotación (la
causa del bloqueo está en el notebook [00](00-jss-corpus-y-auditoria.ipynb), §4.1). Un
kappa de Cohen sobre 3 observaciones, como se verá, casi no dice nada.

## 3.2 Los tres pares y sus etiquetas

In [5]:
pairs = pd.DataFrame(adjudicated_pairs(SERIES, paths=PATHS))
display(
    pairs[
        [
            "pair_id",
            "earlier_decision_id",
            "later_decision_id",
            f"{a}_relation",
            f"{b}_relation",
            "acuerdo",
            "final_relation",
        ]
    ]
)

,pair_id,earlier_decision_id,later_decision_id,gm_relation,gv_relation,acuerdo,final_relation
0,ES2015.p001,ES2015a.d01,ES2015b.d01,refina,refina,si,refina
1,ES2015.p002,ES2015c.d06,ES2015d.d04,reafirma,reafirma,si,reafirma
2,ES2015.p003,ES2015c.d07,ES2015d.d06,reafirma,refina,no,reafirma


## 3.3 Los tres kappas de la Tarea D

El manual (§6) exige calcular **tres kappas independientes**, nunca uno combinado:

- **existencia**: ¿hay o no un vínculo entre las dos decisiones? (`no_relacionada` = no hay);
- **tipo**: qué relación es, calculado solo sobre pares en que *ambos* ven un vínculo;
- **dirección**: si el orden temporal anterior → posterior es el correcto (`direction_ok`).

Se calculan con la misma función que el comando `afg gold agreement`, así que deben
reproducir `reports/tables/ES2015.agreement.csv`.

In [6]:
result = compute_series_agreement(annotator_csvs(PATHS.relations_dir, SERIES, "candidates"))
kappa_table = pd.DataFrame(
    [
        ("existencia", result.n_items, result.existence_kappa, result.existence_disagreements),
        ("tipo", result.n_type_items, result.type_kappa, result.type_disagreements),
        ("dirección", result.n_items, result.direction_kappa, result.direction_disagreements),
    ],
    columns=["eje", "n", "kappa", "pares en desacuerdo"],
)
kappa_table["pares en desacuerdo"] = kappa_table["pares en desacuerdo"].map(", ".join)
display(kappa_table)

# Cross-check against the table already committed by `afg gold agreement`.
committed = pd.read_csv(PROJECT_ROOT / "reports" / "tables" / f"{SERIES}.agreement.csv")
display(committed[["axis", "n_items", "kappa", "disagreeing_pair_ids"]])

,eje,n,kappa,pares en desacuerdo
0,existencia,3,NaN,
1,tipo,3,0.4,ES2015.p003
2,dirección,3,NaN,


,axis,n_items,kappa,disagreeing_pair_ids
0,existence,3,NaN,NaN
1,type,3,0.4,ES2015.p003
2,direction,3,NaN,NaN


## 3.4 Por qué dos de los tres kappas no existen

Dos ejes salen `NaN`: no es un fallo de cálculo, es una propiedad del estadístico. El kappa
vale `(acuerdo observado − acuerdo esperado) / (1 − acuerdo esperado)`. Si **ambos
anotadores usan una sola categoría en todos los ítems**, el acuerdo esperado es 1 y el
denominador se anula: el kappa queda indefinido, aunque el acuerdo crudo sea del 100 %. La
tabla siguiente muestra exactamente eso, eje por eje.

In [7]:
profiles = pd.DataFrame(
    [
        {
            "eje": p.axis,
            "n": p.n_items,
            "acuerdo crudo": p.percent_agreement,
            "categorías usadas (gm)": ", ".join(p.labels_a),
            "categorías usadas (gv)": ", ".join(p.labels_b),
            "kappa definido": p.kappa_defined,
        }
        for p in series_axis_profiles(annotator_csvs(PATHS.relations_dir, SERIES, "candidates"))
    ]
)
display(profiles)

,eje,n,acuerdo crudo,categorías usadas (gm),categorías usadas (gv),kappa definido
0,existence,3,1.000000,exists,exists,False
1,type,3,0.666667,"reafirma, refina","reafirma, refina",True
2,direction,3,1.000000,si,si,False


**Lectura.**

- **Existencia y dirección: acuerdo crudo del 100 %, kappa indefinido.** Los dos anotadores
  vieron un vínculo en los 3 pares y los 3 con la dirección correcta. Sin variación en las
  etiquetas no hay nada que descontar por azar, y el cociente es 0/0. Mostrar "1,0" o "0,0"
  aquí sería inventar un valor.
- **Tipo: kappa 0,40 sobre 3 pares, con un desacuerdo** (`ES2015.p003`: `gm` ve `reafirma`
  y `gv` ve `refina`). Con 3 observaciones, un solo par que cambie mueve el kappa de 0,40 a
  1,0 o a un valor negativo; el número es inestable y no sostiene ninguna conclusión.
- **El diagnóstico útil no es el kappa sino el desacuerdo**: el único par en que difirieron
  es exactamente el borde entre dos etiquetas vecinas (`reafirma` / `refina`), que se
  resuelve en §4.6 y dio lugar al vacío (c) de §5.
- Es un problema que se repetirá en la Fase 2 con las series de pocos candidatos (ES2016 tiene
  4): un kappa por serie sobre tan pocos pares puede salir indefinido o inestable, y hay que
  leerlo con ese límite (notebook 05, pendiente).

---

# 4. Resultado de la adjudicación

`jss` adjudicó solo donde los dos anotadores difieren. Donde coinciden (`acuerdo = si`), el
CSV copia la etiqueta humana común a las columnas `final_*` y registra a `jss` como
adjudicador; **la máquina nunca elige entre dos anotadores que difieren**. Cada veredicto en
un desacuerdo lleva su `razon`, escrita por `jss`. Las razones se muestran tal como están
registradas.

## 4.1 Estados finales

La referencia final distingue las filas base de las filas hijas que nacen de una
`compuesta`.

In [8]:
counts = final_status_counts(SERIES, paths=PATHS)
status_table = (
    pd.DataFrame({"filas base": counts.base, "filas hijas (de compuesta)": counts.children})
    .fillna(0)
    .astype(int)
)
status_table.loc["total"] = status_table.sum()
display(status_table)

,filas base,filas hijas (de compuesta)
compuesta,2,0
decision,23,4
no_decision,3,0
sin_soporte,1,0
descartar,0,1
total,29,5


**Lectura.**

- **De las 29 decisiones base, 23 quedan como `decision`, 3 como `no_decision`, 1 como
  `sin_soporte` y 2 como `compuesta`.** Las `compuesta` no son un veredicto final sino una
  instrucción: se dividen en filas hijas, que sí llevan el estado (§4.2).
- **Las 5 filas hijas añaden 4 decisiones y 1 `descartar`.** La referencia de ES2015 queda,
  por tanto, con 27 filas `decision` (23 base + 4 hijas).
- Los estados `no_decision` se conservan a propósito: miden los falsos positivos de la
  extracción (manual §2).

## 4.2 Las dos filas `compuesta`

Una frase de decisión es `compuesta` cuando reúne varias decisiones **separables** que deben
evaluarse por separado. Aquí aparecen dos casos, y en ambos la `compuesta` salió de un
desacuerdo (una persona la vio y la otra no).

In [9]:
children = pd.DataFrame(compuesta_children(SERIES, paths=PATHS))
display(children[["parent_id", "decision_id", "final_status", "final_object", "final_content"]])
print()
for record in compuesta_children(SERIES, paths=PATHS):
    show_razon(record, ["decision_id", "final_status"])

,parent_id,decision_id,final_status,final_object,final_content
0,ES2015b.d04,ES2015b.d04-1,decision,LCD,dice como conlusion que no LCD
1,ES2015b.d04,ES2015b.d04-2,descartar,Teletex,dice como conlusion que No teletex
2,ES2015b.d04,ES2015b.d04-3,decision,Voice Recognition,dice como conlusion que no reconocimiento por voz
3,ES2015d.d02,ES2015d.d02-1,decision,carcasa intercambiable,tener carcasas intercambiables
4,ES2015d.d02,ES2015d.d02-2,decision,diseno con nombre de frutas y verduras,"los esquemas de color se nombran y colorean según frutas y verduras (p. ej.,..."



decision_id=ES2015b.d04-1 | final_status=decision
    razon: Son 3 exclusiones; el teletexto se descarta por duplicar ES2015b.d02

decision_id=ES2015b.d04-2 | final_status=descartar
    razon: Duplica ES2015b.d02: el teletexto ya está anotado como decisión explícita en esa fila, y la frase
           de ES2015b.d04 no lo menciona; solo aparece en la evidencia.

decision_id=ES2015b.d04-3 | final_status=decision
    razon: Son 3 decisiones de diseno en el contexto de concluir lo conversado

decision_id=ES2015d.d02-1 | final_status=decision
    razon: La carcasa intercambiable ya venía acordada de reuniones anteriores; aquí se mantiene.

decision_id=ES2015d.d02-2 | final_status=decision
    razon: Se plantea de forma tentativa ("could be named"), pero el equipo la adopta: más adelante evalúan el
           producto con modelos llamados banana y kiwi.



**Lectura.**

- **ES2015b.d04** (frase: exclusiones del diseño) se divide en tres exclusiones sobre objetos
  distintos: pantalla LCD, teletexto y reconocimiento de voz. De las tres, la del teletexto
  se descarta por duplicar una fila ya anotada de la misma reunión (§4.3).
- **ES2015d.d02** reúne la *carcasa intercambiable* y los *nombres y colores de frutas y
  verduras*. Son dos decisiones de objeto distinto: una podría cambiar en otra reunión sin
  la otra. La carcasa, además, ya venía acordada de reuniones anteriores y solo se mantiene.
- El criterio que separa estos casos de una frase con un solo objeto se fija en §5 (a).

## 4.3 La decisión duplicada (`descartar`)

La división de ES2015b.d04 dejó una fila hija, `ES2015b.d04-2` (teletexto), que repite una
decisión que la misma reunión ya anota como `ES2015b.d02`. Contar la decisión dos veces
inflaría la reunión y, más adelante, generaría un par de la decisión consigo misma.

In [10]:
for record in final_rows(SERIES, "descartar", paths=PATHS):
    show_razon(record, ["decision_id", "parent_id", "final_status"])

decision_id=ES2015b.d04-2 | parent_id=ES2015b.d04 | final_status=descartar
    razon: Duplica ES2015b.d02: el teletexto ya está anotado como decisión explícita en esa fila, y la frase
           de ES2015b.d04 no lo menciona; solo aparece en la evidencia.



**Lectura.** La regla aplicada: una decisión repetida dentro de la misma reunión **se anota
una sola vez**; se conserva la fila cuya frase la enuncia de forma explícita y la otra pasa a
`descartar`, nombrando en `razon` la fila que duplica (manual §2). Aquí se conserva
`ES2015b.d02` porque su frase nombra el teletexto; en la frase de `ES2015b.d04` el teletexto
solo aparece en la evidencia. La regla es una de las decisiones de §5 (b).

## 4.4 La fila `sin_soporte`

`sin_soporte` significa que la evidencia no respalda lo que la frase afirma. Es el único
caso de ES2015 con veredicto final `sin_soporte`, y es un buen ejemplo de por qué la
transcripción se lee.

In [11]:
for record in final_rows(SERIES, "sin_soporte", paths=PATHS):
    show_razon(record, ["decision_id", "final_status"])
    print(f"frase de decisión (resumen): {record['sentence_text']}")

decision_id=ES2015d.d07 | final_status=sin_soporte
    razon: el costo calculado es 12.20 y no 1.20

frase de decisión (resumen): Product costing is 1.20 Euros.


**Lectura.** El resumen abstractivo dice *"Product costing is 1.20 Euros"*, pero en la
transcripción (`data/interim/transcripts/ES2015d.md`, línea 1185) el gerente de proyecto
dice *"our operating cost is twelve twenty Euros"*, es decir, 12,20 y no 1,20. La frase del
resumen no está respaldada por lo que se dijo. Ambos anotadores coincidieron en
`sin_soporte`, y `jss` lo ratificó con la razón anterior porque, para este estado, la razón
es obligatoria aunque haya acuerdo.

## 4.5 Los desacuerdos de decisiones, uno por uno

Hubo 16 desacuerdos en filas base (13 coincidencias de 29, §2). En la tabla,
`sided_with` indica a qué anotador dio la razón el veredicto final (`ninguno` si no coincide
con ninguno de los dos).

In [12]:
disagreements = pd.DataFrame(resolved_disagreements(SERIES, paths=PATHS))
display(disagreements.drop(columns="razon"))
print("Veredicto final coincide con:", side_taken_counts(SERIES, paths=PATHS))

,decision_id,gm_status,gv_status,final_status,sided_with
0,ES2015a.d01,decision,no_decision,decision,gm
1,ES2015a.d03,decision,no_decision,no_decision,gv
2,ES2015b.d02,decision,no_decision,decision,gm
3,ES2015b.d04,compuesta,decision,compuesta,gm
4,ES2015b.d05,decision,no_decision,decision,gm
5,ES2015b.d06,decision,sin_soporte,decision,gm
6,ES2015b.d08,decision,sin_soporte,decision,gm
7,ES2015b.d09,decision,sin_soporte,no_decision,ninguno
8,ES2015c.d02,decision,sin_soporte,decision,gm
9,ES2015c.d04,decision,sin_soporte,decision,gm


Veredicto final coincide con: {'gm': 13, 'gv': 2, 'ninguno': 1}


Las razones registradas, en el orden del archivo:

In [13]:
for record in resolved_disagreements(SERIES, paths=PATHS):
    show_razon(record, ["decision_id", f"{a}_status", f"{b}_status", "final_status"])

decision_id=ES2015a.d01 | gm_status=decision | gv_status=no_decision | final_status=decision
    razon: Me da a entender que se esta definiendo la creacion de un nuevo producto y en esta reunion se
           define que se quiere un nuevo control remoto, de hecho el PM dice `we want this to be a marketable
           product that can be trendy`

decision_id=ES2015a.d03 | gm_status=decision | gv_status=no_decision | final_status=no_decision
    razon: No es una decision, es una meta financiera

decision_id=ES2015b.d02 | gm_status=decision | gv_status=no_decision | final_status=decision
    razon: es una decision, pues se da un input y se decide en las siguientes linas, creo que es importante
           que si la decision se toma en las lineas posteriores sigue siendo una decision, es parte del
           problema de como se formulan las decision

decision_id=ES2015b.d04 | gm_status=compuesta | gv_status=decision | final_status=compuesta
    razon: Son 3 exclusiones; el teletexto se desc

**Lectura.**

- **El veredicto coincidió con `gm` en 13 casos, con `gv` en 2 y con ninguno en 1**
  (`ES2015b.d09`, que `jss` dejó como `no_decision` porque la decisión "se pospone"). Esto
  es coherente con lo visto en §2: `gm` aceptaba casi todo como decisión y, en la mayoría de
  los desacuerdos, la lectura de la evidencia llevó al veredicto a la misma posición.
- **Los motivos se repiten**, y por eso son útiles para la guía:
  - *"Reafirma"*: una propuesta de un participante que otro acepta en la misma reunión es
    una decisión (`ES2015c.d02`, `ES2015d.d04`, `ES2015d.d05`).
  - *Lectura de la transcripción*: una frase que parece sugerencia puede ser una decisión
    si el diálogo posterior la cierra (`ES2015b.d02`, `ES2015b.d05`).
  - *Meta o sugerencia*: lo que es solo meta financiera o sugerencia no es decisión
    (`ES2015a.d03`, y `ES2015b.d07` donde ambos coincidieron).
- Que el veredicto coincida más con `gm` **no se interpreta como que `gm` anotó mejor**: el
  resultado depende de qué frases tenía ES2015. Lo único que se afirma es que la guía no
  fijaba el umbral y por eso dos lecturas razonables diferían.

## 4.6 Los pares

De los 3 pares, 2 tuvieron acuerdo y 1 desacuerdo.

In [14]:
for record in adjudicated_pairs(SERIES, paths=PATHS):
    if record["acuerdo"] == "no":
        print(
            f"{record['pair_id']}: {a}={record[a + '_relation']}, {b}={record[b + '_relation']}, "
            f"final={record['final_relation']} (coincide con {record['sided_with']})"
        )
        print(f"  {record['earlier_decision_id']} -> {record['later_decision_id']}")
        print(
            textwrap.fill(
                f"razon: {record['razon']}",
                width=110,
                initial_indent="  ",
                subsequent_indent="         ",
            )
        )

ES2015.p003: gm=reafirma, gv=refina, final=reafirma (coincide con gm)
  ES2015c.d07 -> ES2015d.d06
  razon: La reunión d repite el material ya decidido en c


**Lectura.** `ES2015.p003` compara *"Case will be plastic and coated in rubber"* (reunión c)
con *"Will be made of plastic and rubber (rubber on remote for comfort)"* (reunión d). `gm`
leyó `reafirma` y `gv` `refina`; el veredicto es `reafirma`. La reunión d repite el material
ya decidido en c (plástico y goma); la mención *"for comfort"* es una **justificación**, no
un detalle nuevo del contenido de la decisión. Por eso no es `refina`: refinar exige que el
contenido cambie. Es el vacío (c) de §5.

---

# 5. Vacíos de la guía que reveló la calibración

La función de la Fase 1 es encontrar dónde la guía deja dos lecturas razonables. Cada vacío
se resolvió en la adjudicación con una regla. Se registran aquí los tres que dejó ES2015, con
su ubicación en el manual de anotación: las reglas (a) y (b) están en §2 y la regla (c) en
§4, incorporadas al cerrar esta fase y antes de que comience la Fase 2.

| | Vacío | Dónde apareció | Regla adoptada |
|---|---|---|---|
| (a) | ¿Cuándo una frase es `compuesta`? | `ES2015b.d04`, `ES2015d.d02` | Es `compuesta` si reúne **decisiones separables sobre objetos distintos**. Prueba: ¿podría una reunión posterior cambiar una parte sin tocar la otra? Los atributos o las razones de una misma decisión **no** son decisiones separables |
| (b) | ¿Qué se hace con una decisión repetida dentro de la misma reunión? | `ES2015b.d04-2` frente a `ES2015b.d02` | Se anota **una sola vez**: se conserva la fila cuya frase la enuncia de forma explícita y la otra se marca `descartar`, nombrando en `razon` la fila que duplica (manual §2) |
| (c) | ¿`reafirma` o `refina` cuando la posterior solo agrega una justificación? | `ES2015.p003` | Es `reafirma`: una **razón** o motivo agregado no cambia el contenido de la decisión. `refina` exige un cambio de contenido (acotar, precisar, agregar un elemento a lo decidido) |

Algunas observaciones sobre cada regla:

- **(a) es un criterio de separabilidad, no de longitud ni de número de palabras.** Una
  frase larga con un solo objeto y varios atributos sigue siendo una decisión. La prueba del
  cambio independiente evita dividir en exceso, que genera filas sin sentido por separado.
- **(b) evita dos defectos distintos:** contar una decisión de más en la reunión y producir
  un par "decisión frente a sí misma" en la Tarea B. Se estableció en el PR #71 y se ajustó
  la razón de `ES2015b.d04` en el PR #72.
- **(c) es un caso del borde entre etiquetas vecinas**, la señal típica de un vacío de la guía
  y no de un descuido de uno de los anotadores. El ejemplo resuelto de
  [`tareas-jss.md` §6](../docs/anotacion/asignacion/tareas-jss.md) muestra el borde opuesto:
  si la posterior *excluye* alternativas antes abiertas, el contenido cambió por acotación y
  es `refina`. Ambos extremos delimitan el criterio.

Lo que **no** se resuelve con una regla de la guía, y queda para la adjudicación caso a caso,
es decidir si una propuesta aceptada en la conversación llegó a ser decisión (§4.5): esa
frontera depende de la evidencia de cada reunión y por eso `razon` es obligatoria.

---

# 6. Qué cambia para la Fase 2 y qué límites tiene esta fase

## 6.1 Qué se lleva a la Fase 2

- **`gv` y `gm` comienzan la Fase 2 con ES2016**, la primera de las series de control, por el
  orden de menor a mayor número de candidatos (ES2016 → IS1003 → TS3005 → ES2008).
- **Llevan las tres reglas de §5**, ya escritas en el manual de anotación (§2 y §4). El
  umbral de decisión de §4.5 no se convirtió en regla: depende de la evidencia de cada
  reunión y se resuelve en la adjudicación, con `razon` obligatoria.
- **Mantienen la independencia**: no comparan etiquetas antes de la sesión de adjudicación.
- **Los kappas de la Fase 2 sí son reportables.** Son los de las cuatro series de control
  doblemente anotadas (notebook 05, pendiente), y se reportan tal como salgan.

## 6.2 Límites de esta fase

- **Tamaño.** 29 decisiones y 3 pares de una sola serie. Los porcentajes y el kappa de §2
  tienen incertidumbre grande, y los de §3 son indefinidos o inestables.
- **Es diagnóstico.** No se debe citar ningún valor de este notebook como acuerdo entre
  anotadores de OE1 en la tesis; el único kappa reportable es el de la Fase 2.
- **Contenido de desarrollo.** Los ejemplos citan frases de ES2015 porque es una serie de
  desarrollo (ADR 0005). Los notebooks 05 y 06 trabajan sobre series de control y de
  evaluación y mostrarán solo estadísticas agregadas.
- **Una sola adjudicadora.** Que `jss` coincida más con uno de los anotadores no mide la
  calidad de ninguno de los dos: mide la distancia entre dos lecturas y la tercera.

## 6.3 Reproducibilidad

Todo lo que se mostró sale de archivos versionados; no requiere el corpus AMI.

```bash
# Re-ejecutar este notebook de punta a punta
uv run --group notebooks jupyter nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=900 notebooks/04-jss-anotacion-f1-calibracion.ipynb

# Regenerar el acuerdo y verificar la adjudicación (insumos de este notebook)
uv run afg gold agreement --series ES2015
uv run afg gold validate-adjudication --series ES2015
```

Las funciones que calculan cada cifra están en
[`src/afg/annotation/calibration.py`](../src/afg/annotation/calibration.py) y
[`src/afg/annotation/agreement.py`](../src/afg/annotation/agreement.py), con sus pruebas en
`tests/test_calibration.py` y `tests/test_agreement.py`.